# IT Subfield Classifier using SPECTER + XGBoost

This notebook builds a subfield classifier for Information Technology research papers using:

- **SPECTER embeddings** (from Title + Abstract)
- **XGBoost** for classification
- **LabelEncoder** for encoding subfields

**Final IT Subfield Labels:**  
- **CLD:** Cloud Systems  
- **IOTNET:** IoT & Networked Systems  
- **OPS:** IT Operations & Optimization  
- **SEC:** Cybersecurity & Access Control

**Target:**  
Achieve high (>90%) classification accuracy across these four IT subfields.

In [ ]:
import sentence_transformers
import xgboost
import sklearn
import joblib

print("All packages are already installed and ready ✅")

## 1. Load and Prepare IT Subfield Dataset

We load the `IT_subfields.csv` file from the Data/ directory. Each row contains:

- **Title:** Research paper title  
- **Abstract:** Paper abstract  
- **Subfield:** IT subfield label (`CLD`, `IOTNET`, `OPS`, `SEC`)  
- **Link:** Source URL for the paper

In [ ]:
import pandas as pd

# Step 1: Load dataset
df = pd.read_csv("Data/IT_subfields.csv")

# Create input_text for embedding
df["input_text"] = df["Title"].astype(str).str.strip() + " " + df["Abstract"].astype(str).str.strip()

# Show shape and preview only the 4 relevant columns
print(df.shape)
df[["Title", "Abstract", "Subfield", "Link"]].head()

In [ ]:
# Preview the constructed input_text
df["input_text"].head()

## 2. Check Subfield Distribution and Encode Labels

We inspect the number of papers per subfield to detect any class imbalance.  
Next, we use `LabelEncoder` to convert subfield labels into numerical format required for model training.

In [ ]:
from collections import Counter

# Check class distribution
subfield_counts = Counter(df["Subfield"])
print("Subfield Distribution:\n")
for subfield, count in subfield_counts.items():
    print(f"{subfield}: {count}")

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode subfield labels
le = LabelEncoder()
df["subfield_label"] = le.fit_transform(df["Subfield"])

# Preview encoded labels
df[["Subfield", "subfield_label"]].drop_duplicates().sort_values("subfield_label")

## 3. Generate SPECTER Embeddings

We now load the `allenai-specter` model using `sentence-transformers` and generate dense 768-dimensional embeddings for each `text` entry.

These embeddings will be used as features for training the XGBoost classifier.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the pretrained SPECTER model
model = SentenceTransformer("allenai-specter")

# Encode input text (this may take 1–2 minutes)
X_embeddings = model.encode(df["input_text"].tolist(), batch_size=32, show_progress_bar=True)

## 4. Train-Test Split

We split the dataset into training and test sets using an 80/20 stratified split.  
This ensures that each subfield is proportionally represented in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Define features and labels
X = X_embeddings
y = df["subfield_label"].values

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 5. Train XGBoost Classifier

We now train an `XGBClassifier` on the SPECTER embeddings.  
This model uses default hyperparameters and will serve as our baseline.

In [ ]:
from xgboost import XGBClassifier

# Initialize classifier
xgb = XGBClassifier(
    use_label_encoder=False,    # disables legacy label encoding warning
    eval_metric='mlogloss',     # multiclass log-loss
    random_state=42
)

# Train the model
xgb.fit(X_train, y_train)

## 6. Evaluate Model Performance

We evaluate the classifier on the test set using:

- Accuracy
- Precision, Recall, and F1-score (per class)
- Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Predict on test set
y_pred = xgb.predict(X_test)

# Decode labels for readability
label_names = le.classes_

# Print classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_names))

# Show raw confusion matrix
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

## 7. Save Model Artefacts

We save the trained XGBoost model and label encoder using `joblib`.  
This allows us to reuse the classifier later without retraining.

In [ ]:
import joblib
import os

# Save model and label encoder
joblib.dump(xgb, "Artefacts/it_subfield_xgb_model_v2.3.pkl")
joblib.dump(le, "Artefacts/it_subfield_label_encoder_v2.3.pkl")

print("Saved model and label encoder to Artefacts folder ✅")

## Classification Results Summary (v2.3 – SPECTER + XGBoost)

The classifier achieved the following evaluation metrics on the test set:

- **Accuracy:** 83%
- **Macro F1-score:** 0.80
- **Weighted F1-score:** 0.82

### Per-Class F1 Scores:

| Subfield | F1 Score | Comments |
|----------|----------|----------|
| CLD      | 0.87     | High accuracy; well-separated from others |
| IOTNET   | 0.83     | Strong performance, minor confusion with CLD/SEC |
| OPS      | 0.61     | Lower recall; some confusion with IOTNET/CLD (few samples) |
| SEC      | 0.89     | Best performer; strong identity in embedding space |

**Observations:**
- The model is robust for most subfields, but struggles with recall on the rarest class (OPS).
- Most confusion occurs between OPS and IOTNET/CLD, possibly due to overlap in operational and infrastructure language.

---

We now proceed to hyperparameter tuning (v2.4) to improve results, especially for underrepresented subfields.

### Confusion Matrix (Heatmap)

We plot the confusion matrix as a heatmap to visualize where the model is confusing between subfields.

- Darker diagonal = better performance.
- Off-diagonal values = misclassifications.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Create heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – IT Subfield Classifier (v2.3)")
plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning (XGBoost)

We use `GridSearchCV` to optimize key hyperparameters:

- `max_depth`: Maximum tree depth
- `learning_rate`: Step size shrinkage
- `n_estimators`: Number of boosting rounds

The best parameters are selected based on 3-fold cross-validated macro F1-score.

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# Define hyperparameter grid
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200]
}

# Initialize base model (remove deprecated args)
xgb_base = XGBClassifier(eval_metric='mlogloss', random_state=42)

# Grid Search with 3-fold CV
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    verbose=1,
    n_jobs=1  # Avoids BrokenProcessPool in Jupyter
)

# Fit to training data
grid_search.fit(X_train, y_train)

# Show best parameters and best macro F1
print("✅ Best Parameters:", grid_search.best_params_)
print("✅ Best Macro F1 Score (CV average):", grid_search.best_score_)

In [ ]:
# Use best estimator from grid search
xgb_tuned = grid_search.best_estimator_
y_pred_tuned = xgb_tuned.predict(X_test)

# Classification report
print("Classification Report (Tuned XGBoost):\n")
print(classification_report(y_test, y_pred_tuned, target_names=label_names))

# Confusion matrix
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_tuned))

In [ ]:
joblib.dump(xgb_tuned, "Artefacts/it_subfield_xgb_model_v2.4_tuned.pkl")
print("✅ Saved tuned model as v2.4")

### Tuning Summary and Result Analysis

We performed hyperparameter tuning using `GridSearchCV` to find the optimal configuration for the XGBoost classifier on SPECTER embeddings.  
The goal was to improve performance on the IT subfield classification task, especially for lower-performing classes like OPS.

---

#### 🔧 Parameter Grid Used

We searched across:
- `max_depth`: [3, 5, 7] – controls tree depth (model complexity)
- `learning_rate`: [0.05, 0.1, 0.2] – controls how much each tree contributes
- `n_estimators`: [100, 200] – number of boosting rounds

---

#### ✅ Best Parameters Found

{'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200}

---

#### 📊 Final Evaluation on Test Set (v2.4 – Tuned XGBoost)
- **Accuracy**: 0.83  
- **Macro F1-score**: 0.80  
- **Weighted F1-score**: 0.83  

---

#### 🔍 Class-wise F1 Breakdown:

| Subfield | F1 Score | Notes |
|----------|----------|-------|
| **CLD**    | 0.86     | Strong, high precision and recall |
| **IOTNET** | 0.84     | Consistent, slight confusion with CLD and SEC |
| **OPS**    | 0.64     | Lowest recall; frequent confusion with CLD/IOTNET; fewest samples |
| **SEC**    | 0.87     | Best performer; nearly all predictions correct |

---

⸻

### 🧠 Interpretation

- Performance remained stable after tuning, with minimal change from the default model (v2.3).
- Shallow trees (`max_depth = 3`) combined with more boosting rounds (`n_estimators = 200`) helped regularize the model and likely prevented overfitting.
- Subfields like **CLD**, **SEC**, and **IOTNET** perform very well, with clear boundaries in SPECTER embedding space.
- **OPS** remains most challenging, with improvement in precision but still limited recall due to its low sample size and some conceptual overlap.
- Most confusion occurs between **OPS**, **CLD**, and **IOTNET**, which aligns with operational/infrastructure thematic similarity in abstracts.

⸻

### 🧪 Next Steps 

- Gather **more data** for underrepresented classes (OPS) to improve recall  
- Add **custom features** (e.g., operational keyword flags, title-only embeddings) to further separate OPS  
- Explore **ensemble models** (e.g., voting between XGBoost and Logistic Regression) for further stability  
- Fine-tune **transformer-based models** or experiment with domain-adaptive SPECTER/SciBERT embeddings for additional gains  

---

Overall, this tuned model (v2.4) serves as a robust and stable baseline for IT subfield classification using SPECTER + XGBoost.  
Further improvements will likely require more data or richer features, rather than hyperparameter tuning alone.